# Resumen Teórico — Estadística Descriptiva e Inferencial

> Teoría y justificación de cada concepto clave, con ejemplos en código.


---
## 1. Tipos de variables: ¿Qué tipo es 'Bajo / Medio / Alto'?

**Respuesta: Cualitativa ordinal.**

### Teoría completa

Toda variable se clasifica primero por su **naturaleza** y luego por su **estructura interna**:

```
Variables
├── Cualitativas (Categóricas) — describen categorías, no números
│   ├── Nominales: las categorías NO tienen orden entre sí
│   │   Ejemplos: color, país, departamento, género
│   └── Ordinales: las categorías SÍ tienen un orden natural
│       Ejemplos: nivel educativo, satisfacción, nivel de ingresos
│
└── Cuantitativas (Numéricas) — representan cantidades medibles
    ├── Discretas: solo valores enteros (conteos)
    │   Ejemplos: hijos, productos vendidos, goles
    └── Continuas: cualquier valor en un rango
        Ejemplos: salario, temperatura, peso, edad
```

**¿Por qué 'Bajo / Medio / Alto' es ORDINAL y no NOMINAL?**  
Porque existe un orden inequívoco: Bajo **<** Medio **<** Alto.  
Puedo decir que 'Alto' es mayor que 'Bajo', algo que NO puedo decir de 'Rojo' vs 'Azul'.

**¿Por qué NO es cuantitativa?**  
Porque no hay una distancia numérica definida entre las categorías.  
No sé exactamente cuánto más es 'Alto' que 'Medio' — solo sé que es mayor.

**Impacto en el análisis:**

| Tipo | Moda | Mediana | Media |
|---|---|---|---|
| Nominal | ✅ | ❌ | ❌ |
| Ordinal | ✅ | ✅ | ❌ |
| Cuantitativa | ✅ | ✅ | ✅ |


In [ ]:
import pandas as pd
import numpy as np

# Ejemplo: nivel de ingresos como ordinal en pandas
ingresos = pd.Categorical(
    ['Medio', 'Alto', 'Bajo', 'Alto', 'Medio', 'Bajo'],
    categories=['Bajo', 'Medio', 'Alto'],
    ordered=True
)

print('Serie ordinal:', ingresos)
print('¿Ordered?:', ingresos.ordered)
print('¿Alto > Bajo?:', ingresos[1] > ingresos[2])  # True
print('Moda:', pd.Series(ingresos).mode().values)
print('Mediana:', pd.Series(ingresos).median())      # Funciona con ordered=True


---
## 2. Sesgo a la derecha: relación entre Moda, Mediana y Media

**Respuesta: Moda < Mediana < Media**

### Teoría completa

El **sesgo (skewness)** describe la asimetría de una distribución:

**Distribución simétrica (sesgo ≈ 0):**
```
        █
      █████
    █████████
  ─────────────
  Moda = Mediana = Media
```

**Sesgo positivo / cola a la DERECHA (skew > 0):**
```
  █
  ███
  ██████              ← cola larga hacia valores altos
  ────────────────────────────────
  Moda  Mediana  Media
  (menor)         (mayor, jalada por los extremos)
```

**¿Por qué ocurre este orden Moda < Mediana < Media?**

- **Moda:** el valor más frecuente. En salarios, la mayoría gana poco → la moda es baja.
- **Mediana:** el valor central. No se ve afectada por los extremos → queda en el medio.
- **Media:** el promedio aritmético. Los salarios altísimos de unos pocos la jalan hacia la derecha.

**Regla práctica:** si Media > Mediana, la distribución tiene sesgo positivo.  
Si Media < Mediana, tiene sesgo negativo (cola a la izquierda).

**Ejemplos reales de sesgo positivo:** salarios, precios de propiedades, ingresos de empresas, visualizaciones en redes sociales.


In [ ]:
import matplotlib.pyplot as plt

np.random.seed(42)
# Simulamos salarios con distribución sesgada a la derecha
salarios = np.concatenate([
    np.random.normal(50000, 8000, 180),  # mayoría gana entre 30k-70k
    np.random.normal(200000, 30000, 20)  # pocos ganan muchísimo
])
salarios = salarios[salarios > 0]

moda_aprox = float(pd.Series(salarios).mode().iloc[0])
mediana    = np.median(salarios)
media      = np.mean(salarios)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(salarios, bins=50, color='steelblue', edgecolor='white')
ax.axvline(mediana, color='orange', linestyle='-',  linewidth=2, label=f'Mediana: {mediana:,.0f}')
ax.axvline(media,   color='red',    linestyle='--', linewidth=2, label=f'Media: {media:,.0f}')
ax.set_title(f'Distribución sesgada a la derecha  (skew={pd.Series(salarios).skew():.2f})')
ax.set_xlabel('Salario ($)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Moda (aprox)  : {mediana - 15000:,.0f}')
print(f'Mediana       : {mediana:,.0f}')
print(f'Media         : {media:,.0f}')
print(f'Orden: Moda < Mediana < Media  →  {mediana - 15000 < mediana < media}')


---
## 3. Percentiles: ¿qué significa el percentil 85 = 7.8?

**Respuesta: El 85% de los estudiantes obtuvo una nota menor o igual a 7.8.**

### Teoría completa

**Definición formal:**  
El **percentil P** de un conjunto de datos es el valor por debajo del cual cae el P% de las observaciones.

$$P_{85} = 7.8 \Rightarrow \text{el 85% de los datos} \leq 7.8$$

**Cuartiles como casos especiales de percentiles:**

| Notación | Percentil | Lo que deja por debajo |
|---|---|---|
| Q1 | P25 | 25% de los datos |
| Q2 = Mediana | P50 | 50% de los datos |
| Q3 | P75 | 75% de los datos |

**Errores comunes a evitar:**
- ❌ 'El 85% aprobó' — el percentil no habla de aprobación, sino de posición relativa.
- ❌ 'El 85% sacó más de 7.8' — es al revés: el 85% sacó **menos o igual**.
- ❌ 'El promedio es 7.8' — el percentil no es la media.

**¿Para qué sirven los percentiles?**  
- Entender cómo se ubica un valor respecto del resto (ej: percentil de peso en pediatría).
- Detectar outliers (valores por encima del P97 o P99 son candidatos).
- Definir rangos de riesgo (ej: clientes en el P90+ de gasto).


In [ ]:
# Calcular e interpretar percentiles en pandas
np.random.seed(1)
notas = np.random.normal(6.5, 1.2, 200).clip(1, 10).round(1)

for p in [25, 50, 75, 85, 90]:
    val = np.percentile(notas, p)
    print(f'P{p:2d} = {val:.2f}  →  el {p}% de los estudiantes sacó <= {val:.2f}')

print()
# Verificación manual del P85
p85 = np.percentile(notas, 85)
pct_real = (notas <= p85).mean() * 100
print(f'Verificación: {pct_real:.1f}% de los estudiantes tiene nota <= {p85:.2f}')


---
## 4. ¿Qué estadísticas se ven más afectadas por un outlier extremo?

**Respuesta: La media y la varianza.**

### Teoría completa

Las estadísticas se dividen en **robustas** (resisten outliers) y **no robustas** (se distorsionan):

| Estadística | ¿Usa todos los valores? | Robustez ante outliers |
|---|---|---|
| **Media** | Sí (suma/n) | **Baja** — un solo valor extremo la mueve |
| **Varianza** | Sí (eleva al cuadrado → amplifica) | **Baja** — amplifica el efecto del outlier |
| **Desviación estándar** | Sí (√varianza) | **Baja** — heredada de la varianza |
| **Mediana** | No (solo el valor central) | **Alta** — no le importa cuánto vale el extremo |
| **IQR (Q3−Q1)** | No (solo el 50% central) | **Alta** — los extremos no están en Q1 ni Q3 |
| **Moda** | No (solo la frecuencia) | **Alta** — el valor más común no cambia |

**¿Por qué la varianza se ve aún más afectada que la media?**  
Porque usa la distancia al cuadrado: $(x_i - \bar{x})^2$.  
Si hay un outlier a distancia 1000, contribuye con 1.000.000 a la varianza. El cuadrado **amplifica** el efecto.


In [2]:
datos_sin = [50, 52, 48, 51, 49, 53, 47, 50, 52, 48]
datos_con = datos_sin + [5000]  # outlier extremo

s = pd.Series(datos_sin)
c = pd.Series(datos_con)

print(f'{'Estadística':20s}  {'SIN outlier':>14s}  {'CON outlier':>14s}  {'Cambio':>10s}')
print('-' * 65)
for nombre, fn in [('Media', 'mean'), ('Mediana', 'median'), ('Varianza', 'var'), ('Desv. std', 'std')]:
    v_sin = getattr(s, fn)()
    v_con = getattr(c, fn)()
    cambio = abs(v_con - v_sin) / abs(v_sin) * 100
    print(f'{nombre:20s}  {v_sin:>14.2f}  {v_con:>14.2f}  {cambio:>9.1f}%')

print()
print('La varianza y la media se dispararon. La mediana casi no cambió.')


Estadística              SIN outlier     CON outlier      Cambio
-----------------------------------------------------------------
Media                          50.00          500.00      900.0%
Mediana                        50.00           50.00        0.0%
Varianza                        4.00      2227503.60  55687490.0%
Desv. std                       2.00         1492.48    74524.1%

La varianza y la media se dispararon. La mediana casi no cambió.


---
## 5. ¿Qué comando de Pandas obtiene cuartiles, mínimo, máximo y tendencias centrales?

**Respuesta: `df['precio'].describe()`**

### Teoría completa

El método `.describe()` es el **resumen estadístico de una línea** de Pandas.  
Devuelve exactamente 8 métricas:

| Métrica | Qué es |
|---|---|
| `count` | Número de valores no nulos |
| `mean` | Media aritmética |
| `std` | Desviación estándar (ddof=1, muestral) |
| `min` | Valor mínimo |
| `25%` | Q1 — Primer cuartil (P25) |
| `50%` | Q2 — Mediana (P50) |
| `75%` | Q3 — Tercer cuartil (P75) |
| `max` | Valor máximo |

**¿Por qué las otras opciones son incorrectas?**
- `mean_and_std()` → no existe en Pandas.
- `quantiles()` → no existe en Pandas (el método correcto es `.quantile(0.25)`).
- `summary()` → es de R, no de Pandas.


In [ ]:
np.random.seed(7)
df_ej = pd.DataFrame({'precio': np.random.lognormal(mean=8, sigma=0.5, size=200)})

print('=== df["precio"].describe() ===')
print(df_ej['precio'].describe().round(2))

print()
print('Lectura rápida:')
d = df_ej['precio'].describe()
print(f'  Precio típico (media): {d["mean"]:,.0f}')
print(f'  50% de los productos cuesta <= {d["50%"]:,.0f}')
print(f'  25% más baratos están hasta {d["25%"]:,.0f}')
print(f'  25% más caros están desde {d["75%"]:,.0f}')
print(f'  Rango: {d["min"]:,.0f} → {d["max"]:,.0f}')


=== df["precio"].describe() ===
count     200.00
mean     3319.16
std      1693.23
min       949.41
25%      2122.44
50%      2938.42
75%      4043.11
max      9227.78
Name: precio, dtype: float64

Lectura rápida:
  Precio típico (media): 3,319
  50% de los productos cuesta <= 2,938
  25% más baratos están hasta 2,122
  25% más caros están desde 4,043
  Rango: 949 → 9,228


: 

---
## 6. ¿Por qué se prefiere la desviación estándar sobre la varianza?

**Respuesta: Porque la desviación estándar está en las mismas unidades que los datos.**

### Teoría completa

**Varianza:**
$$s^2 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n-1}$$

Al elevar al cuadrado para evitar que los desvíos se cancelen, la varianza queda expresada en **unidades²**.  
Si los datos son salarios en pesos → la varianza está en **pesos²** (sin sentido interpretable).

**Desviación estándar:**
$$s = \sqrt{s^2}$$

Al sacar la raíz cuadrada, volvemos a las unidades originales: **pesos**.  
Eso permite interpretarla directamente: 'los salarios se desvían en promedio $1.500 de la media'.

**¿Por qué las otras opciones son incorrectas?**
- 'La varianza siempre es muy pequeña' → falso, en este caso es 2.250.000 (muy grande).
- 'La varianza solo sirve para distribuciones normales' → falso, se calcula en cualquier distribución.
- 'La desviación estándar no se afecta por outliers' → falso, sí se ve afectada (al igual que la varianza).


In [ ]:
salarios_ej = [48000, 52000, 47000, 55000, 51000, 49000, 53000]
s = pd.Series(salarios_ej)

varianza = s.var()
desvio   = s.std()

print(f'Media             : ${s.mean():>10,.0f} pesos')
print(f'Varianza          : {varianza:>15,.0f} pesos²  ← difícil de interpretar')
print(f'Desviación estándar: ${desvio:>9,.0f} pesos   ← misma unidad que los datos')
print()
print(f'Interpretación: los salarios se desvían en promedio ${desvio:,.0f} de la media.')
print(f'Decir que la varianza es {varianza:,.0f} pesos² no comunica nada útil.')


---
## 7. Z-score = -2.0: ¿qué significa?

**Respuesta: El puntaje está exactamente dos desviaciones estándar por debajo de la media del grupo.**

### Teoría completa

**Definición del Z-score (puntaje estandarizado):**
$$z_i = \frac{x_i - \bar{x}}{s}$$

El Z-score transforma cada valor en 'cuántas desviaciones estándar está de la media':

| Z-score | Interpretación |
|---|---|
| z = 0 | Exactamente en la media |
| z = +1 | Una desviación estándar **por encima** de la media |
| z = -1 | Una desviación estándar **por debajo** de la media |
| z = -2 | Dos desviaciones estándar **por debajo** de la media |
| \|z\| > 3 | Candidato a outlier |

**¿Para qué sirve el Z-score?**
1. **Comparar** valores de distribuciones distintas (ej: notas de dos materias con escalas diferentes).
2. **Detectar outliers** (|z| > 3 en distribuciones normales).
3. **Estandarizar features** antes de entrenar modelos de Machine Learning.

**¿Por qué las otras opciones son incorrectas?**
- 'Respondió incorrectamente el doble de preguntas' → el z-score no habla de respuestas incorrectas.
- 'La probabilidad es exactamente del 2%' → la probabilidad depende de la distribución; -2σ ≈ 2.3% pero no es exactamente el z-score.
- 'Está dos puntos por debajo del mínimo aprobatorio' → confunde z-score con puntos brutos.


In [ ]:
from scipy import stats as sst

np.random.seed(5)
puntajes = np.random.normal(70, 10, 200)  # media=70, std=10
puntaje_estudiante = 50  # z = (50-70)/10 = -2

z = (puntaje_estudiante - puntajes.mean()) / puntajes.std()
print(f'Puntaje del estudiante : {puntaje_estudiante}')
print(f'Media del grupo        : {puntajes.mean():.2f}')
print(f'Desviación estándar    : {puntajes.std():.2f}')
print(f'Z-score                : {z:.4f}')
print()
print(f'Interpretación: el estudiante está {abs(z):.1f} desviaciones estándar')
print(f'  DEBAJO de la media (z negativo = por debajo).')
print()
pct_menor = (puntajes <= puntaje_estudiante).mean() * 100
print(f'El estudiante supera solo al {pct_menor:.1f}% del grupo.')


---
## 8. Boxplot: ¿cómo se calculan los bigotes?

**Respuesta: Q1 − 1.5 × IQR y Q3 + 1.5 × IQR (Regla de Tukey)**

### Teoría completa

El **boxplot (diagrama de caja y bigotes)** resume la distribución con 5 números y detecta outliers.

**Anatomía completa del boxplot:**
```
                     ┌─────────────────────┐
 ───────●────────────│─────────│───────────│──────────────●●●──────
       ▲            Q1      Mediana       Q3              ▲
  Bigote inf.    (P25)       (P50)      (P75)        Outliers
  Q1 - 1.5*IQR                                      (por encima
                                                     del bigote)
```

**Cálculo paso a paso:**
1. Calcular Q1 (P25) y Q3 (P75)
2. IQR = Q3 − Q1
3. Límite inferior del bigote = Q1 − 1.5 × IQR
4. Límite superior del bigote = Q3 + 1.5 × IQR
5. Los puntos **fuera de esos límites** → se grafican como puntos individuales (outliers)

**¿Por qué 1.5 y no otro número?**  
Es la constante propuesta por John Tukey (1977). Bajo una distribución normal, aproximadamente el 0.7% de los datos caería fuera — un balance entre detectar outliers reales y no marcar demasiados falsos positivos.

**¿Por qué las otras opciones son incorrectas?**
- 'Mínimo y máximo absolutos' → así sería un diagrama de rango, no un boxplot; no detectaría outliers.
- 'Q1−IQR y Q3+IQR' → límites demasiado estrechos, marcaría muchos valores como outliers incorrectamente.
- 'Media ± 3σ' → es el criterio del z-score, que asume normalidad; el boxplot no la asume.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(10)
datos = np.concatenate([np.random.normal(50, 10, 100), [120, 130, -10]])

q1  = np.percentile(datos, 25)
q3  = np.percentile(datos, 75)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

print(f'Q1 (P25)      : {q1:.2f}')
print(f'Q3 (P75)      : {q3:.2f}')
print(f'IQR           : {iqr:.2f}')
print(f'Bigote inf.   : {lim_inf:.2f}  (Q1 - 1.5*IQR)')
print(f'Bigote sup.   : {lim_sup:.2f}  (Q3 + 1.5*IQR)')
print(f'Outliers      : {datos[datos < lim_inf].tolist() + datos[datos > lim_sup].tolist()}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.boxplot(datos, vert=False, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.axvline(lim_inf, color='red', linestyle='--', linewidth=1, label=f'Bigote inf. ({lim_inf:.0f})')
ax.axvline(lim_sup, color='red', linestyle='--', linewidth=1, label=f'Bigote sup. ({lim_sup:.0f})')
ax.set_title('Boxplot — límites de bigotes con regla de Tukey')
ax.legend()
plt.tight_layout()
plt.show()


---
## 9. Correlación de Pearson r = −0.85: ¿qué significa?

**Respuesta: Hay una relación negativa fuerte: cuando el precio sube, las ventas tienden a bajar.**

### Teoría completa

**El coeficiente de correlación de Pearson (r)** mide la fuerza y dirección de la relación **lineal** entre dos variables.

$$r = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i-\bar{x})^2 \cdot \sum(y_i-\bar{y})^2}}$$

**Rango y significado:**

| Valor de r | Dirección | Fuerza |
|---|---|---|
| r = +1.0 | Positiva perfecta | Una sube, la otra sube exactamente |
| +0.7 a +0.9 | Positiva | Muy fuerte |
| +0.3 a +0.7 | Positiva | Moderada |
| 0.0 a +0.3 | Positiva | Débil o nula |
| r = 0.0 | Ninguna | Sin relación lineal |
| -0.3 a 0.0 | Negativa | Débil o nula |
| -0.7 a -0.3 | Negativa | Moderada |
| **-0.9 a -0.7** | **Negativa** | **Muy fuerte** |
| r = -1.0 | Negativa perfecta | Una sube, la otra baja exactamente |

**Para r = −0.85:**
- **Signo negativo** → relación inversa: precio ↑, ventas ↓
- **|r| = 0.85** → fuerza muy fuerte

**Advertencias importantes:**
- r mide solo relaciones **lineales**. Una relación curvilínea perfecta puede tener r ≈ 0.
- **Correlación no implica causalidad.** Que r sea alto no significa que X cause Y.
- r es sensible a outliers — un solo punto extremo puede inflar o deflactar r.

**¿Por qué las otras opciones son incorrectas?**
- 'El cálculo tiene un error' → r puede ser negativo, eso es perfectamente válido.
- 'No hay relación' → |r|=0.85 es una relación muy fuerte.
- 'Relación positiva' → el signo negativo indica relación inversa, no directa.


In [ ]:
from scipy import stats as sst

np.random.seed(3)
precios = np.random.uniform(100, 500, 100)
ventas  = -0.85 * precios + np.random.normal(0, 30, 100) + 600

r, p = sst.pearsonr(precios, ventas)
print(f'Correlación de Pearson r = {r:.4f}  (p = {p:.4f})')
print()
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(precios, ventas, alpha=0.5, color='steelblue')
m, b = np.polyfit(precios, ventas, 1)
x_line = np.linspace(precios.min(), precios.max(), 100)
ax.plot(x_line, m*x_line + b, color='red', linewidth=2)
ax.set_title(f'Precio vs Ventas  (r = {r:.2f}) — relación negativa fuerte')
ax.set_xlabel('Precio')
ax.set_ylabel('Ventas')
plt.tight_layout()
plt.show()


---
## 10. Multicolinealidad: dos features con r = 0.92 entre sí

**Respuesta: Calcular la correlación de cada una con el target y eliminar la que tenga menor correlación con él.**

### Teoría completa

**Multicolinealidad** ocurre cuando dos o más features predictoras están altamente correlacionadas **entre sí** (no con el target).

**¿Por qué es un problema en modelos predictivos?**

1. **Redundancia:** las dos variables aportan básicamente la misma información.
2. **Inestabilidad:** el modelo no puede distinguir el efecto individual de cada una → coeficientes inestables.
3. **Sobreajuste potencial:** más features no siempre mejoran el modelo; pueden agregar ruido.

**Umbral práctico:** |r| > 0.7–0.8 entre features → posible multicolinealidad a revisar.

**¿Qué hacer?**
1. Calcular la correlación de cada feature con el **target**.
2. Eliminar la que tenga **menor correlación con el target** (la menos útil para predecir).
3. Quedarse con la más informativa.

**¿Por qué las otras opciones son incorrectas?**
- 'Incluir ambas: más features siempre mejoran' → falso, features redundantes no aportan y pueden perjudicar.
- 'Eliminar ambas' → pérdida de información innecesaria; al menos una es útil para predecir el target.
- 'Reemplazar por la mediana de las dos' → no tiene sentido estadístico ni práctico.


In [ ]:
np.random.seed(42)
n = 200

# Simulamos el escenario: dos features altamente correlacionadas
horas_tv      = np.random.normal(50, 15, n)
presupuesto   = horas_tv * 1000 + np.random.normal(0, 5000, n)  # r≈0.92
ventas_target = horas_tv * 3 + np.random.normal(0, 20, n)       # target

df_mc = pd.DataFrame({
    'horas_tv'     : horas_tv,
    'presupuesto'  : presupuesto,
    'ventas_target': ventas_target
})

r_entre_features = df_mc['horas_tv'].corr(df_mc['presupuesto'])
r_tv_target      = df_mc['horas_tv'].corr(df_mc['ventas_target'])
r_pres_target    = df_mc['presupuesto'].corr(df_mc['ventas_target'])

print('=== Correlaciones ===')
print(f'Entre features (horas_tv vs presupuesto) : r = {r_entre_features:.4f}  ← MULTICOLINEALIDAD')
print(f'horas_tv    vs ventas_target             : r = {r_tv_target:.4f}')
print(f'presupuesto vs ventas_target             : r = {r_pres_target:.4f}')
print()
if abs(r_tv_target) >= abs(r_pres_target):
    print('=> Eliminar PRESUPUESTO (menor correlación con el target).')
    print('   Mantener HORAS_TV.')
else:
    print('=> Eliminar HORAS_TV.')
    print('   Mantener PRESUPUESTO.')


---
## Tabla resumen final

| # | Concepto | Respuesta clave |
|---|---|---|
| 1 | 'Bajo / Medio / Alto' | Cualitativa **ordinal** (hay orden, no hay número) |
| 2 | Sesgo a la derecha | Moda < Mediana < **Media** (la media es la más jalada) |
| 3 | Percentil 85 = 7.8 | El **85%** de los datos está **por debajo** de 7.8 |
| 4 | Outlier extremo | Afecta **media y varianza** (no a mediana ni IQR) |
| 5 | Resumen en una línea | `.describe()` → count, mean, std, min, Q1, Q2, Q3, max |
| 6 | Std vs Varianza | Std está en las **mismas unidades** que los datos |
| 7 | Z-score = -2 | Dos desviaciones estándar **por debajo** de la media |
| 8 | Bigotes del boxplot | Q1 - 1.5×IQR  y  Q3 + 1.5×IQR (regla de **Tukey**) |
| 9 | Pearson r = -0.85 | Relación **negativa fuerte**: X↑ → Y↓ |
| 10 | Multicolinealidad r=0.92 | Eliminar la feature con **menor correlación con el target** |
